In [41]:
# testing bag of words cleaning and characterization. Consider: bigrams, lowercase, lemitization (room/rooms), sparse terms

# start from pre-filtered english reviews

import pandas as pd

df = pd.read_csv("../data/data_eng.csv", index_col=0)
print(df.head())

     Unnamed: 0 Review_Language City Name    Hotel Name  \
985           0         English    Ankara  Hotel Ickale   
986           1         English    Ankara  Hotel Ickale   
987           2         English    Ankara  Hotel Ickale   
988           3         English    Ankara  Hotel Ickale   
989           4         English    Ankara  Hotel Ickale   

                                         Upside_Review  \
985  Staff. Serkan türk was a great help during our...   
986  Nicely furnished room and nicely decorated lob...   
987  Very central, near 2 metro stations. Room was ...   
988  Lovely interior.. great location.. friendly staff   
989      Thanks to Mr.Serkan, our problems were solved   

                                       Downside_Review  Review_Score  \
985                                                NaN          10.0   
986                                                NaN           9.0   
987  The wifi had annoying login that required logg...           9.0   
988  wha

In [42]:
# Confirm the provided NA flags match the real missing values
print(df[["na_up_review", "na_down_review"]].dtypes)
print((df["na_up_review"] == df["Upside_Review"].isna()).all())
print((df["na_down_review"] == df["Downside_Review"].isna()).all())


na_up_review      bool
na_down_review    bool
dtype: object
True
False


In [43]:
# Noted there are some errors in downside reviews
up_mismatch = df[df["na_up_review"] != df["Upside_Review"].isna()]
down_mismatch = df[df["na_down_review"] != df["Downside_Review"].isna()]

print(len(up_mismatch), len(down_mismatch))
print(up_mismatch[["Upside_Review", "na_up_review", "source"]].head(20))
print(down_mismatch[["Downside_Review", "na_down_review", "source"]].head(20))

0 5
Empty DataFrame
Columns: [Upside_Review, na_up_review, source]
Index: []
      Downside_Review  na_down_review  source
11016             NaN           False       1
11116             NaN           False       1
11118             NaN           False       1
11419             NaN           False       1
11838             NaN           False       1


In [44]:
up_empty = df["Upside_Review"].fillna("").str.strip() == ""
down_empty = df["Downside_Review"].fillna("").str.strip() == ""

print((df["na_up_review"] == up_empty).all())
print((df["na_down_review"] == down_empty).all())

True
False


In [45]:
still_off = df[df["na_down_review"] != down_empty]
print(len(still_off))
print(still_off[["Downside_Review", "na_down_review", "source"]].apply(
    lambda col: col.map(repr) if col.name == "Downside_Review" else col))

5
      Downside_Review  na_down_review  source
11016             nan           False       1
11116             nan           False       1
11118             nan           False       1
11419             nan           False       1
11838             nan           False       1


In [46]:
# compute own flags
df["up_empty"] = df["Upside_Review"].fillna("").str.strip() == ""
df["down_empty"] = df["Downside_Review"].fillna("").str.strip() == ""

In [ ]:
# Fill, combine, clean
df[["Upside_Review", "Downside_Review"]] = df[["Upside_Review", "Downside_Review"]].fillna("")
df["text"] = (df["Upside_Review"] + " " + df["Downside_Review"]) \
    .str.lower() \
    .str.replace(r"\s+", " ", regex=True) \
    .str.strip()

# Characterization checks
# 0 = human, 1 = AI
print(df["source"].value_counts())                                  

# note: using self computed fields after noting mismatch
print(df.groupby("source")[["up_empty", "down_empty"]].mean())
print(pd.crosstab(df["source"], df["Sentiment"]))